# Collaborative Filtering Benchmark (Simplified)

**Goal:** Compare 4 similarity measures  
(Cosine, Adjusted‑Cosine, Pearson, Jaccard) for both **user‑based** and **item‑based** memory CF on MovieLens *ml‑latest‑small*.

Metrics  
* Rating prediction: **RMSE**, **MAE**  
* Top‑N quality: **Precision**, **Recall**, **F1**

Just run the notebook top‑to‑bottom. Tune parameters in the final cell.


In [66]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, KNNBasic, SVD, accuracy
from surprise.model_selection import train_test_split
from sklearn.metrics.pairwise import pairwise_distances, cosine_similarity
from collections import defaultdict

In [67]:
def load_data(path, drop_ts=True):
    df = pd.read_csv(path)
    
    # Drop timestamp if present
    if drop_ts and "timestamp" in df.columns:
        df = df.drop(columns=["timestamp"])
    
    # Filter out movies with fewer than 5 ratings
    item_counts = df['movieId'].value_counts()
    popular_items = item_counts[item_counts >= 5].index
    df = df[df['movieId'].isin(popular_items)]
    
    # Load into Surprise Dataset
    reader = Reader(rating_scale=(1, 5))
    data = Dataset.load_from_df(df[['userId', 'movieId', 'rating']], reader)
    
    return df, data

In [68]:
def evaluate(predictions, threshold=4.0, top_n=10):
    rmse = accuracy.rmse(predictions, verbose=False)
    mae = accuracy.mae(predictions, verbose=False)
    
    user_relevant = defaultdict(set)
    user_preds = defaultdict(list)
    
    for uid, iid, true_r, pred_r, _ in predictions:
        if true_r >= threshold:
            user_relevant[uid].add(iid)
        user_preds[uid].append((iid, pred_r))
    
    precisions, recalls = [], []
    for uid in user_preds:
        rec_items = [iid for (iid, _) in sorted(user_preds[uid], key=lambda x: x[1], reverse=True)[:top_n]]
        relevant = user_relevant.get(uid, set())
        if not relevant:
            continue
        tp = len(set(rec_items) & relevant)
        precisions.append(tp / len(rec_items))
        recalls.append(tp / len(relevant))
    
    avg_precision = np.mean(precisions) if precisions else 0
    avg_recall = np.mean(recalls) if recalls else 0
    f1 = (2 * avg_precision * avg_recall) / (avg_precision + avg_recall) if (avg_precision + avg_recall) else 0
    
    return rmse, mae, avg_precision, avg_recall, f1

In [69]:
def adjusted_cosine_predictions(trainset, testset, k, user_based=True):
    n_users, n_items = trainset.n_users, trainset.n_items

    # Build rating matrix
    rating_matrix = np.full((n_users, n_items), np.nan)
    for u in range(n_users):
        for i, r in trainset.ur[u]:
            rating_matrix[u, i] = r

    if user_based:
        # Subtract user means
        user_means = np.nanmean(rating_matrix, axis=1).reshape(-1, 1)
        norm_ratings = rating_matrix - user_means
        norm_ratings[np.isnan(norm_ratings)] = 0
        sim_matrix = cosine_similarity(norm_ratings)
    else:
        # Subtract item means
        item_means = np.nanmean(rating_matrix, axis=0).reshape(1, -1)
        norm_ratings = rating_matrix - item_means
        norm_ratings[np.isnan(norm_ratings)] = 0
        sim_matrix = cosine_similarity(norm_ratings.T)

    predictions = []
    for uid, iid, true_r in testset:
        try:
            u_inner = trainset.to_inner_uid(uid)
            i_inner = trainset.to_inner_iid(iid)
        except ValueError:
            predictions.append((uid, iid, true_r, None, {}))
            continue

        neighbors = []
        if user_based:
            for v_inner, r in trainset.ir[i_inner]:  # all users who rated item
                if v_inner == u_inner:
                    continue
                sim_score = sim_matrix[u_inner, v_inner]
                neighbors.append((sim_score, r))
        else:
            for j_inner, r in trainset.ur[u_inner]:  # all items rated by user
                if j_inner == i_inner:
                    continue
                sim_score = sim_matrix[i_inner, j_inner]
                neighbors.append((sim_score, r))

        neighbors.sort(reverse=True, key=lambda x: x[0])
        neighbors = neighbors[:k]

        sum_sim = sum_r = 0
        for sim_score, r in neighbors:
            sum_sim += abs(sim_score)
            sum_r += sim_score * r

        pred = sum_r / sum_sim if sum_sim > 0 else trainset.global_mean
        predictions.append((uid, iid, true_r, pred, {}))

    return [p for p in predictions if p[3] is not None]


In [70]:
def jaccard_predictions(trainset, testset, k, user_based=True):
    if user_based:
        binary_matrix = np.zeros((trainset.n_users, trainset.n_items))
        for u in range(trainset.n_users):
            for i, _ in trainset.ur[u]:
                binary_matrix[u, i] = 1
        jaccard_sim = 1 - pairwise_distances(binary_matrix, metric='jaccard')
    else:
        binary_matrix = np.zeros((trainset.n_items, trainset.n_users))
        for i in range(trainset.n_items):
            for u, _ in trainset.ir[i]:
                binary_matrix[i, u] = 1
        jaccard_sim = 1 - pairwise_distances(binary_matrix, metric='jaccard')

    predictions = []

    for uid, iid, true_r in testset:
        try:
            u_inner = trainset.to_inner_uid(uid)
            i_inner = trainset.to_inner_iid(iid)
        except ValueError:
            predictions.append((uid, iid, true_r, None, {}))
            continue

        neighbors = []
        if user_based:
            for v_inner, r in trainset.ir[i_inner]:  # all users who rated item
                if v_inner == u_inner:
                    continue
                sim_score = jaccard_sim[u_inner, v_inner]
                neighbors.append((sim_score, r))
        else:
            for j_inner, r in trainset.ur[u_inner]:  # all items rated by user
                if j_inner == i_inner:
                    continue
                sim_score = jaccard_sim[i_inner, j_inner]
                neighbors.append((sim_score, r))

        neighbors.sort(reverse=True, key=lambda x: x[0])
        neighbors = neighbors[:k]

        sum_sim = sum_r = 0
        for sim_score, r in neighbors:
            sum_sim += sim_score
            sum_r += sim_score * r

        pred = sum_r / sum_sim if sum_sim > 0 else trainset.global_mean
        predictions.append((uid, iid, true_r, pred, {}))

    return [p for p in predictions if p[3] is not None]


In [71]:
def run_cf_experiments(trainset, testset, similarities, k_values):
    results = []

    for sim in similarities:
        for k in k_values:
            print(f"Running: sim={sim['name']}, user_based={sim['user_based']}, k={k}")

            if sim['name'] == 'jaccard':
                preds = jaccard_predictions(trainset, testset, k, user_based=sim['user_based'])
            elif sim['name'] == 'adjusted_cosine':
                preds = adjusted_cosine_predictions(trainset, testset, k, user_based=sim['user_based'])
            else:
                algo_cls = KNNBasic
                algo = algo_cls(k=k, sim_options=sim)
                algo.fit(trainset)
                preds = algo.test(testset)

            rmse, mae, prec, rec, f1 = evaluate(preds)
            results.append({
                'similarity': sim['name'],
                'user_based': sim['user_based'],
                'k': k,
                'RMSE': rmse,
                'MAE': mae,
                'Precision': prec,
                'Recall': rec,
                'F1': f1
            })

    return pd.DataFrame(results)


In [72]:
def run_funksvd(trainset, testset, n_factors_list, label="FunkSVD"):
    results = []

    for n_factors in n_factors_list:
        algo = SVD(n_factors=n_factors, random_state=42)
        algo.fit(trainset)
        preds = algo.test(testset)
        rmse, mae, prec, rec, f1 = evaluate(preds)
        results.append({
            'model': label,
            'n_factors': n_factors,
            'RMSE': rmse,
            'MAE': mae,
            'Precision': prec,
            'Recall': rec,
            'F1': f1
        })

    return pd.DataFrame(results)

In [ ]:
similarities = [
    {'name': 'cosine', 'user_based': True},
    {'name': 'cosine', 'user_based': False},   # Adjusted cosine
    {'name': 'adjusted_cosine', 'user_based': True},
    {'name': 'adjusted_cosine', 'user_based': False},
    {'name': 'pearson', 'user_based': True},
    {'name': 'pearson', 'user_based': False},
    {'name': 'jaccard', 'user_based': True},
    {'name': 'jaccard', 'user_based': False},
    {'name': 'msd', 'user_based': True},
    {'name': 'msd', 'user_based': False},
]
k_values = [5, 10, 20, 30, 50, 100, 150]
n_factors_list = [10, 20, 50, 100, 150]

In [74]:
df, data = load_data("data/ratings.csv")
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
print(f"Loaded {len(df):,} ratings from {df['userId'].nunique()} users and {df['movieId'].nunique()} movies")

Loaded 90,274 ratings from 610 users and 3650 movies


In [75]:
cf_results = run_cf_experiments(trainset, testset, similarities, k_values)
cf_results.to_csv('results/cf_experiment_results.csv', index=False)
print(cf_results)

Running: sim=cosine, user_based=True, k=5
Computing the cosine similarity matrix...
Done computing similarity matrix.
Running: sim=cosine, user_based=True, k=10
Computing the cosine similarity matrix...
Done computing similarity matrix.
Running: sim=cosine, user_based=True, k=20
Computing the cosine similarity matrix...
Done computing similarity matrix.
Running: sim=cosine, user_based=True, k=30
Computing the cosine similarity matrix...
Done computing similarity matrix.
Running: sim=cosine, user_based=True, k=50
Computing the cosine similarity matrix...
Done computing similarity matrix.
Running: sim=cosine, user_based=True, k=100
Computing the cosine similarity matrix...
Done computing similarity matrix.
Running: sim=cosine, user_based=True, k=150
Computing the cosine similarity matrix...
Done computing similarity matrix.
Running: sim=cosine, user_based=False, k=5
Computing the cosine similarity matrix...
Done computing similarity matrix.
Running: sim=cosine, user_based=False, k=10
Com

c:\Users\victo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


Running: sim=jaccard, user_based=True, k=10


c:\Users\victo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


Running: sim=jaccard, user_based=True, k=20


...

Running: sim=jaccard, user_based=False, k=100


c:\Users\victo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


Running: sim=jaccard, user_based=False, k=150


c:\Users\victo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


Running: sim=msd, user_based=True, k=5
Computing the msd similarity matrix...
Done computing similarity matrix.
Running: sim=msd, user_based=True, k=10
Computing the msd similarity matrix...
Done computing similarity matrix.
Running: sim=msd, user_based=True, k=20
Computing the msd similarity matrix...
Done computing similarity matrix.
Running: sim=msd, user_based=True, k=30
Computing the msd similarity matrix...
Done computing similarity matrix.
Running: sim=msd, user_based=True, k=50
Computing the msd similarity matrix...
Done computing similarity matrix.
Running: sim=msd, user_based=True, k=100
Computing the msd similarity matrix...
Done computing similarity matrix.
Running: sim=msd, user_based=True, k=150
Computing the msd similarity matrix...
Done computing similarity matrix.
Running: sim=msd, user_based=False, k=5
Computing the msd similarity matrix...
Done computing similarity matrix.
Running: sim=msd, user_based=False, k=10
Computing the msd similarity matrix...
Done computing 

In [76]:
funk_results = run_funksvd(trainset, testset, n_factors_list)
funk_results.to_csv('results/funk_model_experiment_results.csv', index=False)
print(funk_results)

     model  n_factors      RMSE       MAE  Precision    Recall        F1
0  FunkSVD         10  0.849093  0.652629   0.674796  0.687237  0.680960
1  FunkSVD         20  0.850123  0.654468   0.673599  0.686390  0.679935
2  FunkSVD         50  0.849421  0.652574   0.671377  0.684695  0.677971
3  FunkSVD        100  0.852791  0.655539   0.672574  0.685591  0.679020
4  FunkSVD        150  0.855590  0.656793   0.668813  0.685591  0.677098


In [77]:
### --- Sparse experiment --- ###
sparse_df = df.sample(frac=0.2, random_state=42)
_, sparse_data = load_data("data/ratings.csv", drop_ts=True)
sparse_data.raw_ratings = [(str(u), str(i), float(r), 0) for u, i, r in zip(sparse_df['userId'], sparse_df['movieId'], sparse_df['rating'])]
sparse_trainset, sparse_testset = train_test_split(sparse_data, test_size=0.2, random_state=42)

print(f"Sparse data: {len(sparse_df):,} ratings from {sparse_df['userId'].nunique()} users and {sparse_df['movieId'].nunique()} movies")

sparse_cf_results = run_cf_experiments(sparse_trainset, sparse_testset, similarities, k_values)
sparse_cf_results.to_csv('results/cf_sparcity_experiment_results.csv', index=False)
print(sparse_cf_results)

sparse_funk_results = run_funksvd(sparse_trainset, sparse_testset, n_factors_list, label="FunkSVD (sparse)")
sparse_funk_results.to_csv('results/funk_model_experiment_results_sparse.csv', index=False)
print(sparse_funk_results)

Sparse data: 18,055 ratings from 610 users and 3271 movies
Running: sim=cosine, user_based=True, k=5
Computing the cosine similarity matrix...
Done computing similarity matrix.
Running: sim=cosine, user_based=True, k=10
Computing the cosine similarity matrix...
Done computing similarity matrix.
Running: sim=cosine, user_based=True, k=20
Computing the cosine similarity matrix...
Done computing similarity matrix.
Running: sim=cosine, user_based=True, k=30
Computing the cosine similarity matrix...
Done computing similarity matrix.
Running: sim=cosine, user_based=True, k=50
Computing the cosine similarity matrix...
Done computing similarity matrix.
Running: sim=cosine, user_based=True, k=100
Computing the cosine similarity matrix...
Done computing similarity matrix.
Running: sim=cosine, user_based=True, k=150
Computing the cosine similarity matrix...
Done computing similarity matrix.
Running: sim=cosine, user_based=False, k=5
Computing the cosine similarity matrix...
Done computing similar

c:\Users\victo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


Running: sim=jaccard, user_based=True, k=10


c:\Users\victo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


Running: sim=jaccard, user_based=True, k=20


...

Running: sim=jaccard, user_based=False, k=100


c:\Users\victo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


Running: sim=jaccard, user_based=False, k=150


c:\Users\victo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


Running: sim=msd, user_based=True, k=5
Computing the msd similarity matrix...
Done computing similarity matrix.
Running: sim=msd, user_based=True, k=10
Computing the msd similarity matrix...
Done computing similarity matrix.
Running: sim=msd, user_based=True, k=20
Computing the msd similarity matrix...
Done computing similarity matrix.
Running: sim=msd, user_based=True, k=30
Computing the msd similarity matrix...
Done computing similarity matrix.
Running: sim=msd, user_based=True, k=50
Computing the msd similarity matrix...
Done computing similarity matrix.
Running: sim=msd, user_based=True, k=100
Computing the msd similarity matrix...
Done computing similarity matrix.
Running: sim=msd, user_based=True, k=150
Computing the msd similarity matrix...
Done computing similarity matrix.
Running: sim=msd, user_based=False, k=5
Computing the msd similarity matrix...
Done computing similarity matrix.
Running: sim=msd, user_based=False, k=10
Computing the msd similarity matrix...
Done computing 